### Allscripts SCM - Device Exposure Hydration (Surgery Data Mart)

**Source Tables:**
- `_exponent._bronze_srgry_dmart.rpt_case_mtrl_use_vw` - Material/device usage per surgical case
- `_exponent._bronze_srgry_dmart.rpt_mtrl_vw` - Material/device details
- `_exponent._bronze_srgry_dmart.rpt_mfc_vw` - Manufacturer info
- `_exponent._bronze_srgry_dmart.rpt_ptnt_idn_arr_vw` - Patient identifiers

**Strategy:**
- Extract patient_id from MTRL_USE_BK field using REGEXP_EXTRACT
- Link to rpt_mtrl_vw for device/material details
- Use CASE_STRT_TS/CASE_END_TS for device exposure dates
- Map material names to OMOP device concepts
- Use MFC_PART_NUM or MFC_MODL_NUM as unique_device_id

**Note:**
- MTRL_USE_BK format: `SIS5.3~patient_id~{patient_id}~care_event_counter~...~material_id~{material_id}`
- MTRL_USE_FCT_ID is the unique identifier for each device usage record

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW device_exposure_surgery AS
SELECT DISTINCT
  CONCAT_WS(
    chr(31),
    'allscripts_scm',
    'rpt_case_mtrl_use_vw',
    'mtrl_use_fct_id',
    CAST(mtrl_use.MTRL_USE_FCT_ID AS STRING)
  ) AS device_exposure_source_value,

  source_to_person.person_id AS person_id,

  -- Device concept mapping (0 for now, will need concept mapping)
  CAST(0 AS INT) AS device_concept_id,

  -- Device exposure dates from surgical case
  CAST(mtrl_use.CASE_STRT_TS AS DATE) AS device_exposure_start_date,
  mtrl_use.CASE_STRT_TS AS device_exposure_start_datetime,
  CAST(mtrl_use.CASE_END_TS AS DATE) AS device_exposure_end_date,
  mtrl_use.CASE_END_TS AS device_exposure_end_datetime,

  CAST(32817 AS INT) AS device_type_concept_id, -- EHR
  
  -- Unique device identifier from manufacturer
  COALESCE(mtrl.MFC_PART_NUM, mtrl.MFC_MODL_NUM) AS unique_device_id,
  
  -- Production ID (could use vendor part number)
  mtrl.VNDR_PART_NUM AS production_id,
  
  -- Quantity from material usage
  CAST(COALESCE(mtrl_use.MTRL_QTY_NUM, 1) AS DOUBLE) AS quantity,
  
  -- Provider (not available in this data)
  CAST(NULL AS BIGINT) AS provider_id,
  
  -- Visit occurrence (link via case if possible)
  source_to_visit_occurrence.visit_occurrence_id AS visit_occurrence_id,
  CAST(NULL AS BIGINT) AS visit_detail_id,

  -- Device source value - material name or description
  COALESCE(mtrl.MTRL_NM, mtrl.MTRL_DESC, mtrl.PPLSOFT_ITM_NUM) AS device_source_value,

  CAST(0 AS INT) AS device_source_concept_id,

  -- Unit fields
  CAST(NULL AS INT) AS unit_concept_id,
  mtrl.UNIT_OF_ISSU_DESC AS unit_source_value,
  CAST(NULL AS INT) AS unit_source_concept_id,

  'allscripts_scm' AS source_system,

  -- Additional fields for reference
  mtrl_use.CASE_CFRM_NUM AS case_confirmation_number,
  mtrl_use.LOC_NM AS location_name,
  mtrl.MTRL_TP_DESC AS material_type,
  mtrl_use.CHRG_AMT AS charge_amount

FROM _exponent._bronze_srgry_dmart.rpt_case_mtrl_use_vw AS mtrl_use

-- Join to material details
LEFT JOIN _exponent._bronze_srgry_dmart.rpt_mtrl_vw AS mtrl
  ON mtrl.MTRL_DIM_ID = mtrl_use.MTRL_DIM_ID

-- Extract patient_id from MTRL_USE_BK and join to person mapping
-- Format: SIS5.3~patient_id~{patient_id}~care_event_counter~...
JOIN _exponent.omop_mapping.source_to_person AS source_to_person
  ON source_to_person.person_source_value = CONCAT_WS(
       chr(31),
       'allscripts_scm',
       'rpt_ptnt_vw',
       'ptnt_dim_id',
       REGEXP_EXTRACT(mtrl_use.MTRL_USE_BK, 'patient_id~(\\d+)', 1)
     )
  AND source_to_person.active_flag = TRUE

-- Optional: Link to visit occurrence via case
LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence AS source_to_visit_occurrence
  ON source_to_visit_occurrence.visit_occurrence_source_value = CONCAT_WS(
       chr(31),
       'allscripts_scm',
       'rpt_case_vw',
       'case_cfrm_num',
       mtrl_use.CASE_CFRM_NUM
     )
  AND source_to_visit_occurrence.active_flag = TRUE

WHERE mtrl_use.MTRL_USE_FCT_ID IS NOT NULL
  AND mtrl_use.MTRL_USE_BK IS NOT NULL
  AND REGEXP_EXTRACT(mtrl_use.MTRL_USE_BK, 'patient_id~(\\d+)', 1) IS NOT NULL
  AND source_to_person.person_id IS NOT NULL
  AND mtrl_use.CASE_STRT_TS IS NOT NULL
  AND mtrl_use.ISRT_UDT_TS BETWEEN CURRENT_TIMESTAMP() - INTERVAL 365 DAYS AND CURRENT_TIMESTAMP();

In [ ]:
%sql
-- Preview the extracted data
SELECT * FROM device_exposure_surgery LIMIT 20;

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW device_exposure_silver AS
SELECT
  device_exposure_source_value,
  person_id,
  device_concept_id,
  device_exposure_start_date,
  device_exposure_start_datetime,
  device_exposure_end_date,
  device_exposure_end_datetime,
  device_type_concept_id,
  unique_device_id,
  production_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  device_source_value,
  device_source_concept_id,
  unit_concept_id,
  unit_source_value,
  unit_source_concept_id,
  source_system
FROM device_exposure_surgery;

In [ ]:
%sql
MERGE INTO _exponent.omop_silver.device_exposure AS target
USING device_exposure_silver AS source
ON target.device_exposure_source_value = source.device_exposure_source_value

WHEN MATCHED AND NOT (
     target.person_id <=> source.person_id
 AND target.device_concept_id <=> source.device_concept_id
 AND target.device_exposure_start_date <=> source.device_exposure_start_date
 AND target.device_exposure_start_datetime <=> source.device_exposure_start_datetime
 AND target.device_exposure_end_date <=> source.device_exposure_end_date
 AND target.device_exposure_end_datetime <=> source.device_exposure_end_datetime
 AND target.device_type_concept_id <=> source.device_type_concept_id
 AND target.unique_device_id <=> source.unique_device_id
 AND target.production_id <=> source.production_id
 AND target.quantity <=> source.quantity
 AND target.provider_id <=> source.provider_id
 AND target.visit_occurrence_id <=> source.visit_occurrence_id
 AND target.visit_detail_id <=> source.visit_detail_id
 AND target.device_source_value <=> source.device_source_value
 AND target.device_source_concept_id <=> source.device_source_concept_id
 AND target.unit_concept_id <=> source.unit_concept_id
 AND target.unit_source_value <=> source.unit_source_value
 AND target.unit_source_concept_id <=> source.unit_source_concept_id
 AND target.source_system <=> source.source_system
) THEN UPDATE SET
  target.person_id = source.person_id,
  target.device_concept_id = source.device_concept_id,
  target.device_exposure_start_date = source.device_exposure_start_date,
  target.device_exposure_start_datetime = source.device_exposure_start_datetime,
  target.device_exposure_end_date = source.device_exposure_end_date,
  target.device_exposure_end_datetime = source.device_exposure_end_datetime,
  target.device_type_concept_id = source.device_type_concept_id,
  target.unique_device_id = source.unique_device_id,
  target.production_id = source.production_id,
  target.quantity = source.quantity,
  target.provider_id = source.provider_id,
  target.visit_occurrence_id = source.visit_occurrence_id,
  target.visit_detail_id = source.visit_detail_id,
  target.device_source_value = source.device_source_value,
  target.device_source_concept_id = source.device_source_concept_id,
  target.unit_concept_id = source.unit_concept_id,
  target.unit_source_value = source.unit_source_value,
  target.unit_source_concept_id = source.unit_source_concept_id,
  target.source_system = source.source_system,
  target.last_mod_tsp = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  device_exposure_source_value,
  person_id,
  device_concept_id,
  device_exposure_start_date,
  device_exposure_start_datetime,
  device_exposure_end_date,
  device_exposure_end_datetime,
  device_type_concept_id,
  unique_device_id,
  production_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  device_source_value,
  device_source_concept_id,
  unit_concept_id,
  unit_source_value,
  unit_source_concept_id,
  source_system,
  last_mod_tsp
) VALUES (
  source.device_exposure_source_value,
  source.person_id,
  source.device_concept_id,
  source.device_exposure_start_date,
  source.device_exposure_start_datetime,
  source.device_exposure_end_date,
  source.device_exposure_end_datetime,
  source.device_type_concept_id,
  source.unique_device_id,
  source.production_id,
  source.quantity,
  source.provider_id,
  source.visit_occurrence_id,
  source.visit_detail_id,
  source.device_source_value,
  source.device_source_concept_id,
  source.unit_concept_id,
  source.unit_source_value,
  source.unit_source_concept_id,
  source.source_system,
  CURRENT_TIMESTAMP()
);

In [ ]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_device_exposure (
  source_system,
  device_exposure_source_value,
  active_flag,
  created_tsp,
  last_mod_tsp,
  merge_id,
  merge_reason
)
SELECT
  source_distinct.source_system,
  source_distinct.device_exposure_source_value,
  TRUE AS active_flag,
  CURRENT_TIMESTAMP() AS created_tsp,
  COALESCE(source_distinct.last_mod_tsp, CURRENT_TIMESTAMP()) AS last_mod_tsp,
  NULL AS merge_id,
  NULL AS merge_reason
FROM (
  SELECT DISTINCT
    source_system,
    device_exposure_source_value,
    last_mod_tsp
  FROM _exponent.omop_silver.device_exposure
  WHERE device_exposure_source_value IS NOT NULL
    AND source_system = 'allscripts_scm'
) source_distinct
LEFT ANTI JOIN _exponent.omop_mapping.source_to_device_exposure existing
  ON source_distinct.device_exposure_source_value = existing.device_exposure_source_value;

In [ ]:
%sql
-- Preview data before gold merge
SELECT
    source_to_device_exposure.device_exposure_id,
    device_exposure.device_exposure_source_value,
    device_exposure.person_id,
    device_exposure.device_concept_id,
    device_exposure.device_exposure_start_date,
    device_exposure.device_exposure_start_datetime,
    device_exposure.device_exposure_end_date,
    device_exposure.device_exposure_end_datetime,
    device_exposure.device_type_concept_id,
    device_exposure.unique_device_id,
    device_exposure.production_id,
    device_exposure.quantity,
    device_exposure.provider_id,
    device_exposure.visit_occurrence_id,
    device_exposure.visit_detail_id,
    device_exposure.device_source_value,
    device_exposure.device_source_concept_id,
    device_exposure.unit_concept_id,
    device_exposure.unit_source_value,
    device_exposure.unit_source_concept_id,
    device_exposure.source_system,
    device_exposure.last_mod_tsp
  FROM _exponent.omop_silver.device_exposure
  JOIN _exponent.omop_mapping.source_to_device_exposure
    ON device_exposure.device_exposure_source_value = source_to_device_exposure.device_exposure_source_value
   AND source_to_device_exposure.active_flag = TRUE
  WHERE 1=1
  AND device_exposure.source_system = 'allscripts_scm'

In [ ]:
%sql
MERGE INTO _exponent.omop_scm.device_exposure AS target
USING (
  SELECT
    source_to_device_exposure.device_exposure_id,
    device_exposure.device_exposure_source_value,
    device_exposure.person_id,
    device_exposure.device_concept_id,
    device_exposure.device_exposure_start_date,
    device_exposure.device_exposure_start_datetime,
    device_exposure.device_exposure_end_date,
    device_exposure.device_exposure_end_datetime,
    device_exposure.device_type_concept_id,
    device_exposure.unique_device_id,
    device_exposure.production_id,
    device_exposure.quantity,
    device_exposure.provider_id,
    device_exposure.visit_occurrence_id,
    device_exposure.visit_detail_id,
    device_exposure.device_source_value,
    device_exposure.device_source_concept_id,
    device_exposure.unit_concept_id,
    device_exposure.unit_source_value,
    device_exposure.unit_source_concept_id,
    device_exposure.source_system,
    device_exposure.last_mod_tsp
  FROM _exponent.omop_silver.device_exposure
  JOIN _exponent.omop_mapping.source_to_device_exposure
    ON device_exposure.device_exposure_source_value = source_to_device_exposure.device_exposure_source_value
   AND source_to_device_exposure.active_flag = TRUE
  WHERE 1=1
  AND device_exposure.source_system = 'allscripts_scm'
) AS source
ON target.device_exposure_id = source.device_exposure_id

WHEN MATCHED AND NOT (
     target.person_id <=> source.person_id
 AND target.device_concept_id <=> source.device_concept_id
 AND target.device_exposure_start_date <=> source.device_exposure_start_date
 AND target.device_exposure_start_datetime <=> source.device_exposure_start_datetime
 AND target.device_exposure_end_date <=> source.device_exposure_end_date
 AND target.device_exposure_end_datetime <=> source.device_exposure_end_datetime
 AND target.device_type_concept_id <=> source.device_type_concept_id
 AND target.unique_device_id <=> source.unique_device_id
 AND target.production_id <=> source.production_id
 AND target.quantity <=> source.quantity
 AND target.provider_id <=> source.provider_id
 AND target.visit_occurrence_id <=> source.visit_occurrence_id
 AND target.visit_detail_id <=> source.visit_detail_id
 AND target.device_source_value <=> source.device_source_value
 AND target.device_source_concept_id <=> source.device_source_concept_id
 AND target.unit_concept_id <=> source.unit_concept_id
 AND target.unit_source_value <=> source.unit_source_value
 AND target.unit_source_concept_id <=> source.unit_source_concept_id
) THEN UPDATE SET
  target.person_id = source.person_id,
  target.device_concept_id = source.device_concept_id,
  target.device_exposure_start_date = source.device_exposure_start_date,
  target.device_exposure_start_datetime = source.device_exposure_start_datetime,
  target.device_exposure_end_date = source.device_exposure_end_date,
  target.device_exposure_end_datetime = source.device_exposure_end_datetime,
  target.device_type_concept_id = source.device_type_concept_id,
  target.unique_device_id = source.unique_device_id,
  target.production_id = source.production_id,
  target.quantity = source.quantity,
  target.provider_id = source.provider_id,
  target.visit_occurrence_id = source.visit_occurrence_id,
  target.visit_detail_id = source.visit_detail_id,
  target.device_source_value = source.device_source_value,
  target.device_source_concept_id = source.device_source_concept_id,
  target.unit_concept_id = source.unit_concept_id,
  target.unit_source_value = source.unit_source_value,
  target.unit_source_concept_id = source.unit_source_concept_id

WHEN NOT MATCHED THEN INSERT (
  device_exposure_id,
  person_id,
  device_concept_id,
  device_exposure_start_date,
  device_exposure_start_datetime,
  device_exposure_end_date,
  device_exposure_end_datetime,
  device_type_concept_id,
  unique_device_id,
  production_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  device_source_value,
  device_source_concept_id,
  unit_concept_id,
  unit_source_value,
  unit_source_concept_id
) VALUES (
  source.device_exposure_id,
  source.person_id,
  source.device_concept_id,
  source.device_exposure_start_date,
  source.device_exposure_start_datetime,
  source.device_exposure_end_date,
  source.device_exposure_end_datetime,
  source.device_type_concept_id,
  source.unique_device_id,
  source.production_id,
  source.quantity,
  source.provider_id,
  source.visit_occurrence_id,
  source.visit_detail_id,
  source.device_source_value,
  source.device_source_concept_id,
  source.unit_concept_id,
  source.unit_source_value,
  source.unit_source_concept_id
);